# Satellite Upwind Visibility v7 Training

GPU 서버에서 위에서 아래로 실행한다.

- zip 압축 해제
- v7 bundle metadata 검증
- `satellite_only_strong`, `satellite_wind_safe_strong`, `satellite_upwind_visibility_strong` 학습
- clean/fair/real validation 비교
- horizon/region/worst case 결과 출력


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

WORK = Path("/home/j-k14s305/s305-work")
BUNDLE_NAME = "satellite_image_upwind_visibility_regions_2025_20260507_105925"
ZIP = WORK / f"{BUNDLE_NAME}.zip"
OUT = WORK / "runs" / "satellite_upwind_visibility_compare_v7"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4096
EPOCHS = 35
PATIENCE = 8
MIN_DELTA = 1e-4
LR = 3e-4
NUM_WORKERS = 8
FORCE_RETRAIN = False

BASE_NUM_COLS = [
    "cap_scaled",
    "solar_elev_scaled",
    "is_daylight",
    "hour_scaled",
    "doy_scaled",
    "month_scaled",
    "hour_of_day_sin",
    "hour_of_day_cos",
    "day_of_year_sin",
    "day_of_year_cos",
]

WIND_SAFE_COLS = [
    "wind_u_scaled",
    "wind_v_scaled",
    "wind_speed_scaled",
    "wind_dir_sin",
    "wind_dir_cos",
    "asos_ta_scaled",
    "asos_hm_scaled",
    "asos_rn_log1p",
]

VISIBILITY_COLS = [
    "asos_vs_scaled",
    "asos_low_visibility_flag",
    "asos_very_low_visibility_flag",
    "asos_fog_or_mist_flag",
    "asos_haze_code_flag",
]

UPWIND_COLS = [
    "upwind_distance_scaled",
    "upwind_edge_clipped",
    "upwind_ca_scaled",
    "upwind_cf_scaled",
    "upwind_cld_scaled",
    "upwind_cld_ge2_frac",
    "upwind_missing_frac",
    "upwind_center_cf_diff",
    "upwind_center_cld_diff",
]

EXPERIMENTS = [
    ("satellite_only_strong", BASE_NUM_COLS),
    ("satellite_wind_safe_strong", BASE_NUM_COLS + WIND_SAFE_COLS),
    ("satellite_upwind_visibility_strong", BASE_NUM_COLS + WIND_SAFE_COLS + VISIBILITY_COLS + UPWIND_COLS),
]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def resolve_data_dir() -> Path:
    if not ZIP.exists():
        raise FileNotFoundError(ZIP)

    print("zip:", ZIP.exists(), ZIP)
    with zipfile.ZipFile(ZIP, "r") as zf:
        zf.extractall(WORK)

    markers = sorted(WORK.glob("**/metadata/samples_modeling_all_upwind_visibility_v7.parquet"))
    candidates = [m.parents[1] for m in markers if BUNDLE_NAME in str(m.parents[1])]
    if not candidates:
        raise FileNotFoundError("v7 bundle metadata를 찾지 못했다. zip 업로드/압축해제 상태를 확인해야 한다.")

    data_dir = candidates[0]
    print("DATA:", data_dir)
    print("metadata parquet:", len(list((data_dir / "metadata").glob("*.parquet"))))
    print("image shards:", len(list((data_dir / "images").glob("*.npz"))))

    summary_path = data_dir / "metadata" / "upwind_visibility_bundle_summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        print("bundle generated_at:", summary.get("generated_at"))
        print("zip feature count:", len(summary.get("feature_columns", [])))
    return data_dir


seed_everything(SEED)
DATA = resolve_data_dir()

print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
def add_scaled_cols(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["target_timestamp_kst"] = pd.to_datetime(out["target_timestamp_kst"])
    out["cap_scaled"] = (out["estimated_capacity_kw"] / 300000.0).astype("float32")
    out["solar_elev_scaled"] = (out["solar_elevation"] / 90.0).astype("float32")
    out["hour_scaled"] = (out["hour"] / 23.0).astype("float32")
    out["doy_scaled"] = (out["day_of_year"] / 366.0).astype("float32")
    out["month_scaled"] = (out["month"] / 12.0).astype("float32")
    out["is_daylight"] = out["is_daylight"].astype("float32")
    return out


def load_frames():
    train = pd.read_parquet(DATA / "metadata" / "samples_daylight_strong_filter_train.parquet")
    clean_val = pd.read_parquet(DATA / "metadata" / "samples_daylight_strong_filter_val.parquet")
    real_val = pd.read_parquet(DATA / "metadata" / "samples_daylight_no_filter_val.parquet")

    region_map = {name: idx for idx, name in enumerate(sorted(train["region"].unique()))}
    horizon_map = {1: 0, 2: 1, 3: 2, 6: 3}

    frames = []
    for frame in [train, clean_val, real_val]:
        frame = add_scaled_cols(frame)
        frame["region_id"] = frame["region"].map(region_map).astype("int64")
        frame["horizon_id"] = frame["horizon_hours"].astype(int).map(horizon_map).astype("int64")
        frames.append(frame)

    train, clean_val, real_val = frames
    fair_keys = (
        real_val.groupby(["region", "target_timestamp_kst"])["horizon_hours"]
        .nunique()
        .reset_index()
        .query("horizon_hours == 4")[["region", "target_timestamp_kst"]]
    )
    real_fair = real_val.merge(fair_keys, on=["region", "target_timestamp_kst"], how="inner")
    return train, clean_val, real_val, real_fair, region_map, horizon_map


def validate_columns(frame: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [col for col in cols if col not in frame.columns]
    if missing:
        raise KeyError(f"{name} missing columns: {missing}")
    nulls = frame[cols].isna().sum()
    bad = nulls[nulls > 0]
    if len(bad):
        raise ValueError(f"{name} has null model columns:\n{bad}")


train_df, clean_val_df, real_val_df, real_fair_df, region_map, horizon_map = load_frames()
all_model_cols = sorted({col for _, cols in EXPERIMENTS for col in cols})

for frame_name, frame in {
    "train": train_df,
    "clean_val": clean_val_df,
    "real_val": real_val_df,
    "real_fair": real_fair_df,
}.items():
    validate_columns(frame, all_model_cols, frame_name)

print("train:", len(train_df))
print("clean val strong:", len(clean_val_df))
print("real val no_filter:", len(real_val_df))
print("real fair val:", len(real_fair_df), "/", len(real_val_df))
print("regions:", region_map)
print("horizons:", horizon_map)

print("\nfeature ranges:")
print(train_df[all_model_cols].describe().T[["min", "mean", "max"]].to_string())

shards = {}
for path in sorted((DATA / "images").glob("*.npz")):
    shards[path.name] = np.load(path)["images"]

ram_gb = sum(arr.nbytes for arr in shards.values()) / 1024**3
print("\npreloaded image shards:", len(shards), "RAM GB:", round(ram_gb, 2))


In [ ]:
class SolarImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, num_cols: list[str]):
        self.image_file = frame["image_file"].map(lambda value: Path(value).name).to_numpy()
        self.image_row = frame["image_row"].astype("int64").to_numpy()
        self.num = frame[num_cols].astype("float32").to_numpy()
        self.region = frame["region_id"].astype("int64").to_numpy()
        self.horizon = frame["horizon_id"].astype("int64").to_numpy()
        self.y = frame["target_capacity_factor"].astype("float32").to_numpy()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx: int):
        image = shards[self.image_file[idx]][int(self.image_row[idx])].astype("float32", copy=True)
        image[image == 255.0] = 0.0
        image[:, 0] = image[:, 0].clip(0, 1)      # CA is 0/1 in this image bundle
        image[:, 1] = image[:, 1].clip(0, 1)      # CF is 0/1 in this image bundle
        image[:, 2] = image[:, 2] / 9.0           # CT 1..9
        image[:, 3] = image[:, 3] / 3.0           # CLD 0..3, usually 0..2 here
        image = image.reshape(-1, image.shape[-2], image.shape[-1])

        return (
            torch.from_numpy(image),
            torch.from_numpy(self.num[idx]),
            torch.tensor(self.region[idx], dtype=torch.long),
            torch.tensor(self.horizon[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32),
        )


class SatelliteSolarModel(nn.Module):
    def __init__(self, num_dim: int, n_regions: int = 5, n_horizons: int = 4):
        super().__init__()
        self.image = nn.Sequential(
            nn.Conv2d(12, 48, 3, padding=1),
            nn.BatchNorm2d(48),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1),
            nn.BatchNorm2d(96),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(96, 192, 3, padding=1),
            nn.BatchNorm2d(192),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.region = nn.Embedding(n_regions, 8)
        self.horizon = nn.Embedding(n_horizons, 4)
        self.tab = nn.Sequential(
            nn.Linear(num_dim + 12, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(192 + 128, 128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, image, num, region, horizon):
        image_feat = self.image(image).flatten(1)
        tab_input = torch.cat([num, self.region(region), self.horizon(horizon)], dim=1)
        tab_feat = self.tab(tab_input)
        return self.head(torch.cat([image_feat, tab_feat], dim=1)).squeeze(1)


def make_loader(frame: pd.DataFrame, num_cols: list[str], shuffle: bool) -> DataLoader:
    return DataLoader(
        SolarImageDataset(frame, num_cols),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    preds = []
    targets = []
    for image, num, region, horizon, y in loader:
        pred = model(
            image.to(DEVICE, non_blocking=True),
            num.to(DEVICE, non_blocking=True),
            region.to(DEVICE, non_blocking=True),
            horizon.to(DEVICE, non_blocking=True),
        )
        preds.append(pred.detach().float().cpu())
        targets.append(y.detach().float().cpu())

    pred = torch.cat(preds).numpy().clip(0, 1.2)
    y = torch.cat(targets).numpy()
    mae = float(np.mean(np.abs(pred - y)))
    rmse = float(np.sqrt(np.mean((pred - y) ** 2)))
    return mae, rmse, pred, y


In [ ]:
def eval_saved_model(name: str, num_cols: list[str]) -> pd.DataFrame:
    out_dir = OUT / name
    ckpt = torch.load(out_dir / "best_model.pt", map_location=DEVICE, weights_only=False)
    model = SatelliteSolarModel(
        num_dim=len(num_cols),
        n_regions=len(region_map),
        n_horizons=len(horizon_map),
    ).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])

    rows = []
    eval_sets = [
        ("clean_strong_val", clean_val_df),
        ("real_no_filter_fair_val", real_fair_df),
        ("real_no_filter_val", real_val_df),
    ]
    for label, frame in eval_sets:
        loader = make_loader(frame, num_cols, shuffle=False)
        mae, rmse, pred, y = evaluate(model, loader)
        rows.append({"model": name, "eval_set": label, "rows": len(frame), "mae": mae, "rmse": rmse})

        predictions = frame[["target_timestamp_kst", "region", "horizon_hours", "hour", "target_capacity_factor"]].copy()
        predictions["pred"] = pred
        predictions["abs_error"] = np.abs(pred - y)
        predictions.to_csv(out_dir / f"pred_{label}.csv", index=False)

    result = pd.DataFrame(rows)
    result.to_csv(out_dir / "eval_summary.csv", index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def train_one(name: str, num_cols: list[str]) -> pd.DataFrame:
    out_dir = OUT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    if not FORCE_RETRAIN and (out_dir / "best_model.pt").exists() and (out_dir / "eval_summary.csv").exists():
        print("\n==========", name, "==========")
        print("using cached model:", out_dir / "best_model.pt")
        return pd.read_csv(out_dir / "eval_summary.csv")

    train_loader = make_loader(train_df, num_cols, shuffle=True)
    clean_loader = make_loader(clean_val_df, num_cols, shuffle=False)

    model = SatelliteSolarModel(
        num_dim=len(num_cols),
        n_regions=len(region_map),
        n_horizons=len(horizon_map),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    loss_fn = nn.SmoothL1Loss(beta=0.05)

    best_rmse = float("inf")
    bad_epochs = 0
    history = []

    print("\n==========", name, "==========")
    print("num cols:", len(num_cols), num_cols)
    print("params:", round(sum(p.numel() for p in model.parameters()) / 1e6, 3), "M")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for image, num, region, horizon, y in train_loader:
            image = image.to(DEVICE, non_blocking=True)
            num = num.to(DEVICE, non_blocking=True)
            region = region.to(DEVICE, non_blocking=True)
            horizon = horizon.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            pred = model(image, num, region, horizon)
            loss = loss_fn(pred, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            train_loss_sum += float(loss.detach().cpu()) * len(y)
            train_count += len(y)

        scheduler.step()
        train_mae, train_rmse, _, _ = evaluate(model, train_loader)
        val_mae, val_rmse, _, _ = evaluate(model, clean_loader)
        lr = scheduler.get_last_lr()[0]
        train_loss = train_loss_sum / max(train_count, 1)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_mae": train_mae, "train_rmse": train_rmse, "val_mae": val_mae, "val_rmse": val_rmse, "lr": lr})
        print(f"{name} epoch {epoch:02d} | train mae {train_mae:.5f} rmse {train_rmse:.5f} | clean val mae {val_mae:.5f} rmse {val_rmse:.5f} | lr {lr:.7f}")

        if val_rmse < best_rmse - MIN_DELTA:
            best_rmse = val_rmse
            bad_epochs = 0
            torch.save(
                {"model_state": model.state_dict(), "num_cols": num_cols, "region_map": region_map, "horizon_map": horizon_map, "bundle": str(DATA)},
                out_dir / "best_model.pt",
            )
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stop:", epoch)
                break

    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    del model, optimizer, scheduler, train_loader, clean_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    result = eval_saved_model(name, num_cols)
    print("\neval summary:")
    print(result.to_string(index=False))
    return result


In [ ]:
results = []
for name, cols in EXPERIMENTS:
    results.append(train_one(name, cols))

summary = pd.concat(results, ignore_index=True).sort_values(["eval_set", "rmse"])
summary_path = OUT / "model_compare_summary.csv"
summary.to_csv(summary_path, index=False)

print("\n========== FINAL MODEL COMPARISON ==========")
print(summary.to_string(index=False))
print("saved summary:", summary_path)


In [ ]:
summary = pd.read_csv(OUT / "model_compare_summary.csv")

print("OUT exists:", OUT.exists(), OUT)
print("summary exists:", (OUT / "model_compare_summary.csv").exists())
print()
print("===== MODEL SUMMARY =====")
print(summary.sort_values(["eval_set", "rmse"]).to_string(index=False))

pivot_rmse = summary.pivot(index="eval_set", columns="model", values="rmse")
pivot_mae = summary.pivot(index="eval_set", columns="model", values="mae")
print()
print("===== RMSE PIVOT =====")
print(pivot_rmse.to_string())
print()
print("===== MAE PIVOT =====")
print(pivot_mae.to_string())

if "satellite_upwind_visibility_strong" in pivot_rmse.columns and "satellite_wind_safe_strong" in pivot_rmse.columns:
    improvement = (pivot_rmse["satellite_wind_safe_strong"] - pivot_rmse["satellite_upwind_visibility_strong"]) / pivot_rmse["satellite_wind_safe_strong"] * 100
    print()
    print("===== V7 RMSE IMPROVEMENT VS WIND_SAFE (%) =====")
    print(improvement.to_string())


def group_metrics(frame: pd.DataFrame, group_col: str) -> pd.DataFrame:
    work = frame.copy()
    work["_sq_error"] = (work["pred"] - work["target_capacity_factor"]) ** 2
    result = (
        work.groupby(group_col, as_index=False)
        .agg(
            rows=("abs_error", "size"),
            mae=("abs_error", "mean"),
            mse=("_sq_error", "mean"),
            target_mean=("target_capacity_factor", "mean"),
            pred_mean=("pred", "mean"),
        )
    )
    result["rmse"] = np.sqrt(result["mse"])
    result = result.drop(columns=["mse"])
    ordered_cols = [group_col, "rows", "mae", "rmse", "target_mean", "pred_mean"]
    return result[ordered_cols].sort_values("rmse", ascending=False)


for model in [name for name, _ in EXPERIMENTS]:
    print()
    print()
    print("==========", model, "==========")
    model_dir = OUT / model

    hist_path = model_dir / "history.csv"
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
        best = hist.loc[hist["val_rmse"].idxmin()]
        print()
        print("===== BEST EPOCH =====")
        print(best.to_string())
        print()
        print("===== HISTORY TAIL =====")
        print(hist.tail(10).to_string(index=False))

    for eval_set in ["clean_strong_val", "real_no_filter_fair_val", "real_no_filter_val"]:
        p = model_dir / f"pred_{eval_set}.csv"
        if not p.exists():
            print("missing:", p)
            continue

        df = pd.read_csv(p)
        rmse = np.sqrt(((df["pred"] - df["target_capacity_factor"]) ** 2).mean())
        print()
        print("=====", eval_set, "=====")
        print("rows:", len(df))
        print("mae:", df["abs_error"].mean())
        print("rmse:", rmse)
        print()
        print("by horizon")
        print(group_metrics(df, "horizon_hours").to_string(index=False))
        print()
        print("by region")
        print(group_metrics(df, "region").to_string(index=False))
        print()
        print("worst 15")
        print(df.sort_values("abs_error", ascending=False).head(15).to_string(index=False))
